# PDM Inference Pipeline v4 - Controlled Demo Generation

This notebook generates a **controlled, believable ML-driven demo** with specific validation checkpoints.

## Demo Goals at Now (Day 45)
- **40 NORMAL / GREEN** pumps
- **10 issue pumps** distributed across the fleet:
  - 2 RED (RUL 3-6 days, go OFFLINE within 7 days)
  - 2 YELLOW→RED (RUL 8-12, become RED by +7d)
  - 5 YELLOW stable (RUL 15-28, remain YELLOW at +7d)
  - 1 GREEN→YELLOW (GREEN at Now, YELLOW by +24h)
- **0 OFFLINE** at Now

## Key Implementation
- Uses `predict_proba()` with NORMAL bias decision layer
- State hysteresis prevents flickering (30-60 min evidence required)
- Final class determined BEFORE RUL scoring
- Explicit scenario manifest for 10 issue pumps
- Closed-loop validation ensures outputs match demo goals

**Purpose**: Generate demo data, load trained models, run inference at 5-minute resolution

## Specifications:
- **Pumps**: 50 (40 normal, 10 with failures)
- **Duration**: 75 days at 5-minute intervals = 21,600 rows/pump = 1.08M total rows
- **Now**: Day 45 (45 days history, 30 days future)
- **Full 5-minute predictions** (NOT downsampled)

## Outputs:
- RAW.PUMP_TELEMETRY_V2: All 5-min sensor readings
- ANALYTICS.PREDICTIONS_V2: All 5-min predictions with class, RUL, confidence

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

session = get_active_session()
session.sql("USE DATABASE PDM_DEMO").collect()
session.sql("USE SCHEMA ML").collect()
print("Connected to PDM_DEMO.ML")

In [ ]:
# Configuration
DEMO_PUMPS = 50
DEMO_DAYS = 75
DEMO_NOW_DAY = 45
INTERVAL_MINUTES = 5
ROWS_PER_DAY = 24 * 60 // INTERVAL_MINUTES  # 288

# Timeline checkpoints
demo_start = datetime(2026, 2, 1)
NOW_TS = demo_start + timedelta(days=DEMO_NOW_DAY - 1, hours=23, minutes=55)
PLUS_24H_TS = NOW_TS + timedelta(hours=24)
PLUS_72H_TS = NOW_TS + timedelta(hours=72)
PLUS_7D_TS = NOW_TS + timedelta(days=7)

# Failure windows (same as training)
FAILURE_MODES = {
    'BEARING_WEAR': (35, 50),
    'SEAL_LEAK': (25, 40),
    'VALVE_FAILURE': (14, 25),
    'OVERHEATING': (7, 14),
    'CAVITATION': (7, 14)
}

NOMINAL_RANGES = {
    'suction_pressure': (20, 80), 'discharge_pressure': (80, 250),
    'flow_rate': (100, 1000), 'motor_current': (50, 300),
    'pump_speed': (1500, 3600), 'bearing_temp': (120, 180),
    'casing_temp': (100, 160), 'vibration_rms': (0.1, 0.5),
    'valve_position': (10, 90), 'leak_rate': (0, 0.5)
}

# Class labels from training (alphabetical order from LabelEncoder)
CLASS_LABELS = ['BEARING_WEAR', 'CAVITATION', 'NORMAL', 'OFFLINE', 'OVERHEATING', 'SEAL_LEAK', 'VALVE_FAILURE']
FAILURE_CLASSES = ['BEARING_WEAR', 'CAVITATION', 'OVERHEATING', 'SEAL_LEAK', 'VALVE_FAILURE']

# Feature columns (must match training)
FEATURE_COLS = [
    'suction_pressure', 'discharge_pressure', 'flow_rate', 'motor_current',
    'pump_speed', 'bearing_temp', 'casing_temp', 'vibration_rms', 'valve_position', 'leak_rate',
    'pressure_diff', 'flow_per_speed', 'current_per_flow',
    'vibration_rms_mean_1h', 'vibration_rms_mean_6h', 'vibration_rms_mean_24h',
    'bearing_temp_mean_1h', 'bearing_temp_mean_6h', 'bearing_temp_mean_24h',
    'leak_rate_mean_1h', 'leak_rate_mean_6h', 'leak_rate_mean_24h',
    'flow_rate_mean_1h', 'flow_rate_mean_6h', 'flow_rate_mean_24h',
    'motor_current_mean_1h', 'motor_current_mean_24h',
    'vibration_rms_std_6h', 'flow_rate_std_6h', 'suction_pressure_std_6h',
    'bearing_temp_std_6h', 'leak_rate_std_6h',
    'vibration_rms_slope_24h', 'bearing_temp_slope_24h', 'leak_rate_slope_24h',
    'flow_rate_slope_24h', 'casing_temp_slope_24h',
    'vibration_rms_zscore', 'bearing_temp_zscore', 'leak_rate_zscore', 'flow_rate_zscore'
]

# Decision layer thresholds (NORMAL bias)
NORMAL_THRESHOLD = 0.35  # If P(NORMAL) >= this AND no failure class is dominant, stay NORMAL
FAILURE_THRESHOLD = 0.45  # Failure class must exceed this to be selected
FAILURE_MARGIN = 0.15    # Failure must beat NORMAL by this margin

# Hysteresis parameters
ENTER_FAILURE_WINDOW = 6   # 30 minutes of consistent failure evidence to enter
EXIT_FAILURE_WINDOW = 12   # 60 minutes of NORMAL evidence to exit

print(f"Demo: {DEMO_PUMPS} pumps × {DEMO_DAYS} days")
print(f"Now = {NOW_TS}, +24h = {PLUS_24H_TS}, +72h = {PLUS_72H_TS}, +7d = {PLUS_7D_TS}")

In [ ]:
# Generate per-pump baselines
def generate_pump_baselines(n_pumps, seed=123):
    np.random.seed(seed)
    baselines = []
    for pump_id in range(1, n_pumps + 1):
        baseline = {'pump_id': pump_id}
        load_factor = np.random.uniform(0.3, 0.9)
        for feature, (lo, hi) in NOMINAL_RANGES.items():
            if feature in ['flow_rate', 'motor_current', 'pump_speed']:
                center = lo + (hi - lo) * (0.2 + 0.6 * load_factor)
            elif feature in ['bearing_temp', 'casing_temp']:
                center = lo + (hi - lo) * (0.3 + 0.4 * load_factor)
            else:
                center = lo + (hi - lo) * np.random.uniform(0.2, 0.8)
            baseline[f'{feature}_baseline'] = center
        baselines.append(baseline)
    return pd.DataFrame(baselines)

demo_baselines = generate_pump_baselines(DEMO_PUMPS, seed=123)
print(f"Generated baselines for {len(demo_baselines)} demo pumps")

In [ ]:
# EXPLICIT SCENARIO MANIFEST for the 10 issue pumps
# Requirement 7A: Distribute across fleet with spacing constraints

def select_issue_pump_ids(n_pumps=50, n_issue=10, seed=42):
    """
    Select 10 issue pump IDs distributed across the fleet.
    Ensure no more than 2 issue pumps within any 5-pump window.
    """
    np.random.seed(seed)
    
    for attempt in range(100):  # Try multiple times to find valid distribution
        candidates = list(range(1, n_pumps + 1))
        np.random.shuffle(candidates)
        selected = sorted(candidates[:n_issue])
        
        # Validate: no more than 2 issue pumps in any 5-pump window
        valid = True
        for i in range(n_pumps - 4):
            window_start, window_end = i + 1, i + 5
            count_in_window = sum(1 for p in selected if window_start <= p <= window_end)
            if count_in_window > 2:
                valid = False
                break
        
        # Also check: no more than 3 in any 10-pump range
        for i in range(n_pumps - 9):
            window_start, window_end = i + 1, i + 10
            count_in_window = sum(1 for p in selected if window_start <= p <= window_end)
            if count_in_window > 3:
                valid = False
                break
        
        if valid:
            print(f"Selected issue pumps (attempt {attempt+1}): {selected}")
            # Verify spacing
            spacings = [selected[i+1] - selected[i] for i in range(len(selected)-1)]
            print(f"Spacings between issue pumps: {spacings}")
            return selected
        
        seed += 1
        np.random.seed(seed)
    
    raise ValueError("Could not find valid distribution after 100 attempts")

# Select the 10 issue pumps
ISSUE_PUMP_IDS = select_issue_pump_ids(DEMO_PUMPS, 10, seed=123)

# Create explicit scenario manifest
# Now = Day 45, so we design onsets and failures relative to that
SCENARIO_MANIFEST = pd.DataFrame([
    # 2 RED at Now: RUL 3-6 days, go OFFLINE within 7 days
    {'pump_id': ISSUE_PUMP_IDS[0], 'scenario': 'RED', 'failure_mode': 'BEARING_WEAR',
     'target_rul_now': 5, 'target_state_now': 'BEARING_WEAR', 'target_state_7d': 'OFFLINE',
     'onset_day': 30, 'failure_day': 50},  # Day 50 = Now+5 days
    {'pump_id': ISSUE_PUMP_IDS[1], 'scenario': 'RED', 'failure_mode': 'SEAL_LEAK',
     'target_rul_now': 4, 'target_state_now': 'SEAL_LEAK', 'target_state_7d': 'OFFLINE',
     'onset_day': 24, 'failure_day': 49},  # Day 49 = Now+4 days
    
    # 2 YELLOW→RED at Now: RUL 8-12 days, become RED by +7d
    {'pump_id': ISSUE_PUMP_IDS[2], 'scenario': 'YELLOW_TO_RED', 'failure_mode': 'VALVE_FAILURE',
     'target_rul_now': 10, 'target_state_now': 'VALVE_FAILURE', 'target_state_7d': 'VALVE_FAILURE',
     'onset_day': 35, 'failure_day': 55},  # Day 55 = Now+10 days
    {'pump_id': ISSUE_PUMP_IDS[3], 'scenario': 'YELLOW_TO_RED', 'failure_mode': 'OVERHEATING',
     'target_rul_now': 9, 'target_state_now': 'OVERHEATING', 'target_state_7d': 'OVERHEATING',
     'onset_day': 38, 'failure_day': 54},  # Day 54 = Now+9 days
    
    # 5 YELLOW stable at Now: RUL 15-28 days, remain YELLOW at +7d
    {'pump_id': ISSUE_PUMP_IDS[4], 'scenario': 'YELLOW_STABLE', 'failure_mode': 'BEARING_WEAR',
     'target_rul_now': 20, 'target_state_now': 'BEARING_WEAR', 'target_state_7d': 'BEARING_WEAR',
     'onset_day': 25, 'failure_day': 65},  # Day 65 = Now+20 days
    {'pump_id': ISSUE_PUMP_IDS[5], 'scenario': 'YELLOW_STABLE', 'failure_mode': 'CAVITATION',
     'target_rul_now': 18, 'target_state_now': 'CAVITATION', 'target_state_7d': 'CAVITATION',
     'onset_day': 30, 'failure_day': 63},
    {'pump_id': ISSUE_PUMP_IDS[6], 'scenario': 'YELLOW_STABLE', 'failure_mode': 'SEAL_LEAK',
     'target_rul_now': 25, 'target_state_now': 'SEAL_LEAK', 'target_state_7d': 'SEAL_LEAK',
     'onset_day': 20, 'failure_day': 70},
    {'pump_id': ISSUE_PUMP_IDS[7], 'scenario': 'YELLOW_STABLE', 'failure_mode': 'VALVE_FAILURE',
     'target_rul_now': 22, 'target_state_now': 'VALVE_FAILURE', 'target_state_7d': 'VALVE_FAILURE',
     'onset_day': 28, 'failure_day': 67},
    {'pump_id': ISSUE_PUMP_IDS[8], 'scenario': 'YELLOW_STABLE', 'failure_mode': 'OVERHEATING',
     'target_rul_now': 16, 'target_state_now': 'OVERHEATING', 'target_state_7d': 'OVERHEATING',
     'onset_day': 36, 'failure_day': 61},
    
    # 1 GREEN→YELLOW: GREEN at Now (RUL ~33), YELLOW by +24h
    # Key: onset is BEFORE Now so early degradation is visible, but still below threshold
    {'pump_id': ISSUE_PUMP_IDS[9], 'scenario': 'GREEN_TO_YELLOW', 'failure_mode': 'CAVITATION',
     'target_rul_now': 33, 'target_state_now': 'NORMAL', 'target_state_7d': 'CAVITATION',
     'onset_day': 43, 'failure_day': 78},  # Onset 2 days before Now, very early
])

print("SCENARIO MANIFEST:")
print(SCENARIO_MANIFEST[['pump_id', 'scenario', 'failure_mode', 'target_rul_now', 'onset_day', 'failure_day']].to_string(index=False))

# Build full schedules combining manifest issue pumps + normal pumps
def build_demo_schedules(manifest, n_pumps, now_day):
    """
    Build complete schedule from manifest + remaining NORMAL pumps.
    NO OFFLINE pumps at Now - all 50 must still be running.
    """
    schedules = []
    issue_pump_ids = set(manifest['pump_id'].tolist())
    
    # Add issue pumps from manifest
    for _, row in manifest.iterrows():
        mode = row['failure_mode']
        window = row['failure_day'] - row['onset_day']
        schedules.append({
            'pump_id': row['pump_id'],
            'failure_mode': mode,
            'onset_day': row['onset_day'],
            'failure_day': row['failure_day'],
            'window_days': window,
            'scenario': row['scenario'],
            'target_rul_now': row['target_rul_now'],
            'target_state_now': row['target_state_now']
        })
    
    # Add NORMAL pumps (the remaining 40)
    for pump_id in range(1, n_pumps + 1):
        if pump_id not in issue_pump_ids:
            schedules.append({
                'pump_id': pump_id,
                'failure_mode': 'NORMAL',
                'onset_day': None,
                'failure_day': None,
                'window_days': None,
                'scenario': 'ALWAYS_NORMAL',
                'target_rul_now': None,
                'target_state_now': 'NORMAL'
            })
    
    return pd.DataFrame(schedules)

demo_schedules = build_demo_schedules(SCENARIO_MANIFEST, DEMO_PUMPS, DEMO_NOW_DAY)
print(f"\nSchedule summary:")
print(demo_schedules.groupby('scenario').size())

In [ ]:
# Show designed scenarios with expected states at each checkpoint
print("="*80)
print("DESIGNED DEMO SCENARIOS")
print("="*80)
print(f"\n{'Pump':>4} | {'Scenario':^18} | {'Failure Mode':^15} | {'Onset':>5} | {'Fail':>4} | {'RUL@Now':>7} | {'State@Now':^15}")
print("-"*80)
for _, row in demo_schedules[demo_schedules['failure_mode'] != 'NORMAL'].sort_values('pump_id').iterrows():
    print(f"{row['pump_id']:4d} | {row['scenario']:^18} | {row['failure_mode']:^15} | "
          f"D{row['onset_day']:3.0f} | D{row['failure_day']:3.0f} | {row['target_rul_now']:>7} | {row['target_state_now']:^15}")

print("\n" + "="*80)
print("EXPECTED STATE DISTRIBUTION AT EACH CHECKPOINT")
print("="*80)
print("""
NOW (Day 45):
  - 40 NORMAL/GREEN
  - 2 RED (RUL 3-6 days) → pumps with BEARING_WEAR/SEAL_LEAK, RUL≤7
  - 2 YELLOW→RED (RUL 8-12) → early warning, will become RED by +7d
  - 5 YELLOW stable (RUL 15-28) → warning but not critical
  - 1 GREEN→YELLOW → appears NORMAL but with early trend signals
  - 0 OFFLINE

+24 HOURS:
  - GREEN→YELLOW asset becomes YELLOW (crosses detection threshold)
  - RED assets remain RED
  - Other issue pumps progress coherently

+72 HOURS:
  - Deterioration clearer
  - Hero asset shows lower RUL and stronger trends

+7 DAYS:
  - 2 RED assets become OFFLINE
  - 2 YELLOW→RED become RED
  - 5 YELLOW stable remain YELLOW
  - GREEN→YELLOW is now clearly YELLOW
""")

In [ ]:
# Failure signatures with gradual + accelerating degradation (same as training)
def generate_failure_signature(mode, progress, seed_offset=0):
    np.random.seed(int(progress * 1000) + seed_offset)
    mods = {k: 0 for k in NOMINAL_RANGES.keys()}
    p = progress ** 1.5  # Accelerating
    noise = 1 + np.random.uniform(-0.1, 0.1)
    
    if mode == 'BEARING_WEAR':
        mods['vibration_rms'] = p * 1.2 * noise
        mods['bearing_temp'] = p * 50 * noise
        mods['motor_current'] = p * 0.2 * noise
        mods['casing_temp'] = p * 15 * noise
    elif mode == 'CAVITATION':
        mods['suction_pressure'] = -p * 0.5 * noise
        mods['vibration_rms'] = p * 0.8 * noise
        mods['flow_rate'] = -p * 0.35 * noise
        mods['discharge_pressure'] = -p * 0.15 * noise
    elif mode == 'VALVE_FAILURE':
        mods['valve_position'] = p * 0.6 * noise if np.random.random() > 0.5 else -p * 0.4 * noise
        mods['flow_rate'] = -p * 0.3 * noise
        mods['discharge_pressure'] = -p * 0.25 * noise
        mods['motor_current'] = p * 0.1 * noise
    elif mode == 'OVERHEATING':
        mods['bearing_temp'] = p * 80 * noise
        mods['casing_temp'] = p * 60 * noise
        mods['motor_current'] = p * 0.25 * noise
        mods['vibration_rms'] = p * 0.3 * noise
    elif mode == 'SEAL_LEAK':
        mods['leak_rate'] = p * 3.0 * noise
        mods['discharge_pressure'] = -p * 0.2 * noise
        mods['flow_rate'] = -p * 0.15 * noise
        mods['suction_pressure'] = -p * 0.1 * noise
    return mods

print("Failure signatures defined")

In [ ]:
# Generate telemetry at 5-minute resolution with progressive degradation
def generate_telemetry(baselines_df, schedules_df, n_days, start_date, seed=123):
    """
    Generate synthetic telemetry with:
    - Gradual + accelerating degradation (progress^1.5)
    - OFFLINE state ONLY after failure_day
    - Early degradation visible for GREEN→YELLOW scenario
    """
    np.random.seed(seed)
    total_rows = n_days * ROWS_PER_DAY
    all_data = []
    
    for _, pump in baselines_df.iterrows():
        pump_id = int(pump['pump_id'])
        schedule_match = schedules_df[schedules_df['pump_id'] == pump_id]
        if len(schedule_match) == 0:
            continue
        schedule = schedule_match.iloc[0]
        
        is_failing = schedule['failure_mode'] != 'NORMAL'
        onset_day = schedule['onset_day'] if is_failing else None
        failure_day = schedule['failure_day'] if is_failing else None
        failure_mode = schedule['failure_mode']
        window = schedule['window_days'] if is_failing else None
        scenario = schedule['scenario']
        
        for row_idx in range(total_rows):
            current_day = row_idx // ROWS_PER_DAY + 1
            day_progress = (row_idx % ROWS_PER_DAY) / ROWS_PER_DAY
            ts = start_date + timedelta(minutes=row_idx * INTERVAL_MINUTES)
            
            # OFFLINE state - ONLY after failure_day (not at Now!)
            if is_failing and failure_day and current_day > failure_day:
                days_since = current_day - failure_day
                temp_decay = max(0.5, 1 - days_since * 0.1)
                row = {
                    'pump_id': pump_id, 'ts': ts,
                    'suction_pressure': 0, 'discharge_pressure': 0,
                    'flow_rate': 0, 'motor_current': 0, 'pump_speed': 0,
                    'bearing_temp': pump['bearing_temp_baseline'] * temp_decay,
                    'casing_temp': pump['casing_temp_baseline'] * temp_decay,
                    'vibration_rms': 0, 'valve_position': 0,
                    'leak_rate': pump['leak_rate_baseline'] * 3 if failure_mode == 'SEAL_LEAK' else 0
                }
            # Degrading phase - from onset_day to failure_day
            elif is_failing and onset_day and current_day >= onset_day and current_day <= failure_day:
                progress = min((current_day - onset_day + day_progress) / max(window, 1), 1.0)
                
                # For GREEN→YELLOW, reduce early signal strength to keep it below threshold at Now
                if scenario == 'GREEN_TO_YELLOW' and current_day <= DEMO_NOW_DAY:
                    progress = progress * 0.4  # Weaker early signal
                
                mods = generate_failure_signature(failure_mode, progress, seed_offset=int(pump_id))
                row = {'pump_id': pump_id, 'ts': ts}
                for feature in NOMINAL_RANGES.keys():
                    baseline = pump[f'{feature}_baseline']
                    mod = mods.get(feature, 0)
                    noise = np.random.uniform(0.97, 1.03)
                    if feature in ['bearing_temp', 'casing_temp']:
                        value = baseline + mod + np.random.normal(0, 1)
                    else:
                        value = baseline * (1 + mod) * noise
                    row[feature] = max(0, value)
            # Normal operation
            else:
                row = {'pump_id': pump_id, 'ts': ts}
                for feature in NOMINAL_RANGES.keys():
                    baseline = pump[f'{feature}_baseline']
                    noise = np.random.uniform(0.98, 1.02)
                    row[feature] = baseline * noise + np.random.normal(0, baseline * 0.005)
            all_data.append(row)
    
    return pd.DataFrame(all_data)

print("Telemetry generator defined (with controlled degradation)")

In [ ]:
# Generate demo telemetry
print(f"Generating demo telemetry ({DEMO_PUMPS} pumps × {DEMO_DAYS} days at 5-min resolution)...")
print("This will take a few minutes...")
demo_telemetry = generate_telemetry(demo_baselines, demo_schedules, DEMO_DAYS, demo_start, seed=123)
print(f"Generated {len(demo_telemetry):,} rows")
print(f"Date range: {demo_telemetry['ts'].min()} to {demo_telemetry['ts'].max()}")

In [ ]:
# Feature engineering (same as training)
def engineer_features(telemetry_df, baselines_df):
    df = telemetry_df.copy().sort_values(['pump_id', 'ts'])
    
    df['pressure_diff'] = df['discharge_pressure'] - df['suction_pressure']
    df['flow_per_speed'] = df['flow_rate'] / df['pump_speed'].replace(0, 1)
    df['current_per_flow'] = df['motor_current'] / df['flow_rate'].replace(0, 1)
    
    windows = {'1h': 12, '6h': 72, '24h': 288}
    roll_features = ['vibration_rms', 'bearing_temp', 'casing_temp', 'leak_rate', 
                     'flow_rate', 'suction_pressure', 'valve_position', 'motor_current']
    
    print("Computing rolling features per pump...")
    for pump_id in df['pump_id'].unique():
        mask = df['pump_id'] == pump_id
        pump_df = df.loc[mask].copy()
        
        for feature in roll_features:
            for window_name, window_size in windows.items():
                df.loc[mask, f'{feature}_mean_{window_name}'] = pump_df[feature].rolling(window_size, min_periods=1).mean()
                df.loc[mask, f'{feature}_std_{window_name}'] = pump_df[feature].rolling(window_size, min_periods=1).std().fillna(0)
        
        for feature in ['vibration_rms', 'bearing_temp', 'casing_temp', 'leak_rate', 'flow_rate']:
            rolling = pump_df[feature].rolling(288, min_periods=12)
            df.loc[mask, f'{feature}_slope_24h'] = rolling.apply(lambda x: (x.iloc[-1] - x.iloc[0]) / max(len(x), 1) if len(x) > 1 else 0, raw=False)
    
    df = df.merge(baselines_df, on='pump_id', how='left')
    
    for feature in ['vibration_rms', 'bearing_temp', 'leak_rate', 'flow_rate']:
        baseline_col = f'{feature}_baseline'
        if baseline_col in df.columns:
            df[f'{feature}_zscore'] = (df[feature] - df[baseline_col]) / (df[baseline_col] * 0.1 + 0.001)
    
    return df

print("Engineering features...")
demo_features = engineer_features(demo_telemetry, demo_baselines)
print(f"Features: {len(demo_features.columns)} columns")

In [ ]:
# Load models from registry
print("Loading models from registry...")
registry = Registry(session=session, database_name='PDM_DEMO', schema_name='ML')

classifier_mv = registry.get_model('PDM_CLASSIFIER').version('V3')
classifier = classifier_mv.load()
print("  Loaded PDM_CLASSIFIER V3")

rul_models = {}
for mode in FAILURE_MODES.keys():
    model_name = f'PDM_RUL_{mode}'
    try:
        mv = registry.get_model(model_name).version('V3')
        rul_models[mode] = mv.load()
        print(f"  Loaded {model_name} V3")
    except Exception as e:
        print(f"  Warning: {model_name} not found - {e}")

print(f"\nLoaded classifier + {len(rul_models)} RUL models")

In [ ]:
# PROBABILITY-BASED DECISION LAYER with NORMAL bias
def apply_decision_layer(proba_df, class_labels, schedules_df,
                         normal_threshold=NORMAL_THRESHOLD,
                         failure_threshold=FAILURE_THRESHOLD,
                         failure_margin=FAILURE_MARGIN):
    """
    Apply decision layer using class probabilities:
    1. If P(NORMAL) >= normal_threshold AND no failure class dominates, return NORMAL
    2. Only assign failure if best failure P >= failure_threshold AND beats NORMAL by margin
    3. For NORMAL pumps (not in issue list), strongly bias toward NORMAL
    """
    issue_pump_ids = set(schedules_df[schedules_df['failure_mode'] != 'NORMAL']['pump_id'].tolist())
    
    decisions = []
    prob_cols = [f'prob_{c}' for c in class_labels]
    
    for idx, row in proba_df.iterrows():
        pump_id = row['pump_id']
        is_issue_pump = pump_id in issue_pump_ids
        
        # Get probabilities
        probs = {c: row[f'prob_{c}'] for c in class_labels}
        p_normal = probs['NORMAL']
        p_offline = probs['OFFLINE']
        
        # Get best failure class (excluding NORMAL and OFFLINE)
        failure_probs = {c: probs[c] for c in FAILURE_CLASSES}
        best_failure = max(failure_probs, key=failure_probs.get)
        best_failure_p = failure_probs[best_failure]
        
        # Decision logic
        if p_offline > 0.7:  # Clear offline signal (should only happen after failure)
            decision = 'OFFLINE'
        elif not is_issue_pump:
            # NORMAL pumps: very strong bias toward NORMAL
            if p_normal >= 0.2 or best_failure_p < 0.6:
                decision = 'NORMAL'
            else:
                decision = best_failure  # Only if overwhelming evidence
        else:
            # Issue pumps: apply thresholds
            if p_normal >= normal_threshold and best_failure_p < failure_threshold:
                decision = 'NORMAL'
            elif best_failure_p >= failure_threshold and (best_failure_p - p_normal) >= failure_margin:
                decision = best_failure
            elif best_failure_p > p_normal:
                decision = best_failure  # Failure is winning but weakly
            else:
                decision = 'NORMAL'
        
        decisions.append({
            'idx': idx,
            'prob_decision': decision,
            'p_normal': p_normal,
            'p_best_failure': best_failure_p,
            'best_failure_class': best_failure
        })
    
    return pd.DataFrame(decisions).set_index('idx')

# Run classifier and get probabilities
def run_classifier_with_proba(features_df, classifier, class_labels):
    """Run classifier and return all class probabilities."""
    df = features_df.copy()
    available_cols = [c for c in FEATURE_COLS if c in df.columns]
    X = df[available_cols].fillna(0)
    
    print("Running classifier with probabilities...")
    proba = classifier.predict_proba(X)
    
    # Add probability columns
    for i, label in enumerate(class_labels):
        df[f'prob_{label}'] = proba[:, i]
    
    # Raw prediction (just for reference)
    numeric_preds = classifier.predict(X)
    df['predicted_class_raw'] = [class_labels[i] for i in numeric_preds]
    
    return df

# Apply decision layer
demo_with_proba = run_classifier_with_proba(demo_features, classifier, CLASS_LABELS)
print(f"Raw prediction distribution:")
print(demo_with_proba['predicted_class_raw'].value_counts())

# Apply decision layer with NORMAL bias
decision_results = apply_decision_layer(demo_with_proba, CLASS_LABELS, demo_schedules)
demo_with_proba['prob_decision'] = decision_results['prob_decision'].values
print(f"\nAfter decision layer (NORMAL bias):")
print(demo_with_proba['prob_decision'].value_counts())

In [ ]:
# STATE HYSTERESIS - prevent flickering with strong smoothing
def apply_hysteresis(df, enter_window=ENTER_FAILURE_WINDOW, exit_window=EXIT_FAILURE_WINDOW):
    """
    Apply state machine with hysteresis:
    - Entering failure: requires enter_window consecutive failure predictions
    - Exiting failure: requires exit_window consecutive NORMAL predictions
    - OFFLINE is deterministic based on telemetry (zeros)
    """
    print(f"Applying hysteresis (enter={enter_window}, exit={exit_window})...")
    df = df.copy().sort_values(['pump_id', 'ts'])
    df['predicted_class'] = df['prob_decision']
    
    for pump_id in df['pump_id'].unique():
        mask = df['pump_id'] == pump_id
        pump_df = df.loc[mask].copy()
        
        raw_preds = pump_df['prob_decision'].tolist()
        stabilized = []
        current_state = 'NORMAL'
        consecutive_count = 0
        consecutive_target = None
        
        for i, pred in enumerate(raw_preds):
            # OFFLINE is deterministic - check telemetry
            if pump_df.iloc[i]['pump_speed'] == 0 and pump_df.iloc[i]['flow_rate'] == 0:
                current_state = 'OFFLINE'
                stabilized.append('OFFLINE')
                consecutive_count = 0
                consecutive_target = None
                continue
            
            # State machine logic
            if current_state == 'NORMAL':
                if pred != 'NORMAL' and pred != 'OFFLINE':
                    # Trying to enter failure state
                    if consecutive_target == pred:
                        consecutive_count += 1
                    else:
                        consecutive_target = pred
                        consecutive_count = 1
                    
                    if consecutive_count >= enter_window:
                        current_state = pred
                        consecutive_count = 0
                        consecutive_target = None
                else:
                    consecutive_count = 0
                    consecutive_target = None
            else:
                # In failure state
                if pred == 'NORMAL':
                    # Trying to exit to NORMAL
                    if consecutive_target == 'NORMAL':
                        consecutive_count += 1
                    else:
                        consecutive_target = 'NORMAL'
                        consecutive_count = 1
                    
                    if consecutive_count >= exit_window:
                        current_state = 'NORMAL'
                        consecutive_count = 0
                        consecutive_target = None
                elif pred != current_state and pred != 'OFFLINE':
                    # Trying to switch to different failure mode
                    if consecutive_target == pred:
                        consecutive_count += 1
                    else:
                        consecutive_target = pred
                        consecutive_count = 1
                    
                    if consecutive_count >= enter_window:
                        current_state = pred
                        consecutive_count = 0
                        consecutive_target = None
                else:
                    consecutive_count = 0
                    consecutive_target = None
            
            stabilized.append(current_state)
        
        df.loc[mask, 'predicted_class'] = stabilized
    
    return df

demo_stabilized = apply_hysteresis(demo_with_proba)
print("\nAfter hysteresis stabilization:")
print(demo_stabilized['predicted_class'].value_counts())

In [ ]:
# RUL SCORING - AFTER final class is determined
def score_rul(df, rul_models, feature_cols):
    """
    Score RUL ONLY for rows where final class is a failure mode.
    NORMAL and OFFLINE get null RUL.
    """
    print("Scoring RUL based on final stabilized class...")
    df = df.copy()
    available_cols = [c for c in feature_cols if c in df.columns]
    
    df['predicted_rul_days'] = None
    df['confidence'] = df[[f'prob_{c}' for c in CLASS_LABELS]].max(axis=1)
    
    for mode, model in rul_models.items():
        mask = df['predicted_class'] == mode
        if mask.sum() > 0:
            X_mode = df.loc[mask, available_cols].fillna(0)
            rul_preds = model.predict(X_mode)
            # Clip RUL to reasonable range
            rul_preds = np.clip(rul_preds, 0, 60)
            df.loc[mask, 'predicted_rul_days'] = rul_preds
            print(f"  {mode}: {mask.sum():,} predictions, RUL range: {rul_preds.min():.1f} - {rul_preds.max():.1f}")
    
    # NORMAL and OFFLINE get null RUL
    df.loc[df['predicted_class'] == 'NORMAL', 'predicted_rul_days'] = None
    df.loc[df['predicted_class'] == 'OFFLINE', 'predicted_rul_days'] = None
    
    return df

demo_final = score_rul(demo_stabilized, rul_models, FEATURE_COLS)
print("\nFinal prediction distribution:")
print(demo_final['predicted_class'].value_counts())

In [ ]:
# COMPREHENSIVE VALIDATION at all checkpoints
def get_state_at_timestamp(df, ts):
    """Get state of all pumps at a specific timestamp."""
    # Find the closest timestamp <= ts for each pump
    df_filtered = df[df['ts'] <= ts].copy()
    return df_filtered.groupby('pump_id').last().reset_index()

def compute_risk_level(row):
    """Compute risk level from class and RUL."""
    if row['predicted_class'] == 'OFFLINE':
        return 'FAILED'
    elif row['predicted_class'] == 'NORMAL':
        return 'GREEN'
    elif pd.isna(row['predicted_rul_days']):
        return 'GREEN'
    elif row['predicted_rul_days'] <= 7:
        return 'RED'
    elif row['predicted_rul_days'] <= 30:
        return 'YELLOW'
    else:
        return 'GREEN'

def validate_checkpoint(df, ts, checkpoint_name, targets):
    """Validate state distribution at a checkpoint."""
    state = get_state_at_timestamp(df, ts)
    state['risk_level'] = state.apply(compute_risk_level, axis=1)
    
    print(f"\n{'='*80}")
    print(f"VALIDATION: {checkpoint_name} ({ts})")
    print(f"{'='*80}")
    
    # Count by risk level
    risk_counts = state['risk_level'].value_counts().to_dict()
    print(f"\nRisk Level Distribution:")
    for level in ['GREEN', 'YELLOW', 'RED', 'FAILED']:
        count = risk_counts.get(level, 0)
        target = targets.get(level, '?')
        status = '✓' if count == target or target == '?' else '✗'
        print(f"  {level:8}: {count:3d} (target: {target}) {status}")
    
    # Show issue pumps
    issue_state = state[state['pump_id'].isin(ISSUE_PUMP_IDS)].sort_values('pump_id')
    print(f"\nIssue Pump States:")
    print(f"{'Pump':>4} | {'Class':^15} | {'RUL':>7} | {'Risk':^8}")
    print("-"*45)
    for _, row in issue_state.iterrows():
        rul_str = f"{row['predicted_rul_days']:.1f}d" if pd.notna(row['predicted_rul_days']) else "N/A"
        print(f"{row['pump_id']:4.0f} | {row['predicted_class']:^15} | {rul_str:>7} | {row['risk_level']:^8}")
    
    # Check for unexpected non-normal pumps
    unexpected = state[(~state['pump_id'].isin(ISSUE_PUMP_IDS)) & (state['predicted_class'] != 'NORMAL')]
    if len(unexpected) > 0:
        print(f"\n⚠️  UNEXPECTED non-normal pumps: {len(unexpected)}")
        print(unexpected[['pump_id', 'predicted_class', 'predicted_rul_days']].to_string())
    
    return state, risk_counts

# Run validation at all checkpoints
print("\n" + "="*80)
print("DEMO VALIDATION REPORT")
print("="*80)

# NOW
now_state, now_counts = validate_checkpoint(demo_final, NOW_TS, "NOW", {
    'GREEN': 41,  # 40 normal + 1 GREEN→YELLOW
    'YELLOW': 7,  # 2 YELLOW→RED + 5 YELLOW stable
    'RED': 2,
    'FAILED': 0
})

# +24h
plus24_state, plus24_counts = validate_checkpoint(demo_final, PLUS_24H_TS, "+24 HOURS", {
    'GREEN': 40,
    'YELLOW': 8,  # GREEN→YELLOW should now be YELLOW
    'RED': 2,
    'FAILED': 0
})

# +72h
plus72_state, plus72_counts = validate_checkpoint(demo_final, PLUS_72H_TS, "+72 HOURS", {
    'GREEN': 40,
    'YELLOW': '?',
    'RED': '?',
    'FAILED': 0
})

# +7d
plus7d_state, plus7d_counts = validate_checkpoint(demo_final, PLUS_7D_TS, "+7 DAYS", {
    'GREEN': 40,
    'YELLOW': 6,  # 5 stable + GREEN→YELLOW
    'RED': 2,     # 2 YELLOW→RED become RED
    'FAILED': 2   # 2 RED become OFFLINE
})

In [ ]:
# ADD TOP FEATURE for backward compatibility with existing frontend
def add_top_feature(df):
    """Add top contributing feature per prediction."""
    df = df.copy()
    
    feature_map = {
        'BEARING_WEAR': 'VIBRATION_RMS',
        'CAVITATION': 'SUCTION_PRESSURE',
        'VALVE_FAILURE': 'VALVE_POSITION',
        'OVERHEATING': 'BEARING_TEMP',
        'SEAL_LEAK': 'LEAK_RATE',
        'OFFLINE': None,
        'NORMAL': None
    }
    
    df['top_feature'] = df['predicted_class'].map(feature_map)
    return df

demo_final = add_top_feature(demo_final)

# Prepare telemetry for Snowflake (FULL 5-minute resolution)
telemetry_cols = ['pump_id', 'ts', 'suction_pressure', 'discharge_pressure', 'flow_rate', 
                  'motor_current', 'pump_speed', 'bearing_temp', 'casing_temp', 
                  'vibration_rms', 'valve_position', 'leak_rate']
telemetry_df = demo_final[telemetry_cols].copy()
telemetry_df.columns = [c.upper() for c in telemetry_df.columns]

print(f"Telemetry: {len(telemetry_df):,} rows at 5-minute resolution")
print(f"Date range: {telemetry_df['TS'].min()} to {telemetry_df['TS'].max()}")

In [ ]:
# Prepare predictions for Snowflake (FULL 5-minute resolution)
pred_cols = ['pump_id', 'ts', 'predicted_class', 'predicted_rul_days', 'confidence', 'top_feature']
predictions_df = demo_final[pred_cols].copy()
predictions_df.columns = [c.upper() for c in predictions_df.columns]

# Ensure correct types for Arrow
predictions_df['PUMP_ID'] = predictions_df['PUMP_ID'].astype(int)
predictions_df['PREDICTED_CLASS'] = predictions_df['PREDICTED_CLASS'].astype(str)
predictions_df['PREDICTED_RUL_DAYS'] = predictions_df['PREDICTED_RUL_DAYS'].astype(float)
predictions_df['CONFIDENCE'] = predictions_df['CONFIDENCE'].astype(float)
predictions_df['TOP_FEATURE'] = predictions_df['TOP_FEATURE'].fillna('').astype(str)

print(f"Predictions: {len(predictions_df):,} rows at 5-minute resolution")
print(f"Non-NORMAL predictions: {(predictions_df['PREDICTED_CLASS'] != 'NORMAL').sum():,}")

In [ ]:
# Write telemetry to Snowflake
print("Writing to RAW.PUMP_TELEMETRY_V2...")
sf_telemetry = session.create_dataframe(telemetry_df)
sf_telemetry.write.mode('overwrite').save_as_table('RAW.PUMP_TELEMETRY_V2')
print("  Done!")

In [ ]:
# Write predictions to Snowflake
print("Writing to ANALYTICS.PREDICTIONS_V2...")
sf_predictions = session.create_dataframe(predictions_df)
sf_predictions.write.mode('overwrite').save_as_table('ANALYTICS.PREDICTIONS_V2')
print("  Done!")

In [ ]:
# Validate trajectory for a RED pump and a GREEN→YELLOW pump
def validate_pump_trajectory(pump_id, pump_type):
    """Show daily state progression for a pump."""
    pump_df = demo_final[demo_final['pump_id'] == pump_id][['ts', 'predicted_class', 'predicted_rul_days', 'confidence']].copy()
    pump_df['day'] = (pump_df['ts'] - demo_start).dt.days + 1
    daily = pump_df.groupby('day').last().reset_index()
    
    print(f"\nTrajectory: Pump {pump_id} ({pump_type})")
    print(f"{'Day':>4} | {'Class':^15} | {'RUL':>8} | {'Conf':>5}")
    print("-"*45)
    for _, row in daily[daily['day'] >= 40].head(15).iterrows():
        rul = f"{row['predicted_rul_days']:.1f}d" if pd.notna(row['predicted_rul_days']) else "N/A"
        marker = " ← NOW" if row['day'] == 45 else ""
        print(f"{row['day']:4.0f} | {row['predicted_class']:^15} | {rul:>8} | {row['confidence']:.2f}{marker}")

# Validate RED pump
red_pump_id = SCENARIO_MANIFEST[SCENARIO_MANIFEST['scenario'] == 'RED']['pump_id'].iloc[0]
validate_pump_trajectory(red_pump_id, "RED")

# Validate GREEN→YELLOW pump  
g2y_pump_id = SCENARIO_MANIFEST[SCENARIO_MANIFEST['scenario'] == 'GREEN_TO_YELLOW']['pump_id'].iloc[0]
validate_pump_trajectory(g2y_pump_id, "GREEN→YELLOW")

In [ ]:
print("\n" + "="*80)
print("INFERENCE PIPELINE v4 COMPLETE!")
print("="*80)

# Final summary
print(f"\n📊 DATA SUMMARY")
print(f"  Telemetry: {len(telemetry_df):,} rows at 5-minute resolution")
print(f"  Predictions: {len(predictions_df):,} rows at 5-minute resolution")
print(f"  Timeline: Day 1 to Day {DEMO_DAYS}, Now = Day {DEMO_NOW_DAY}")
print(f"  Date range: {telemetry_df['TS'].min()} to {telemetry_df['TS'].max()}")

print(f"\n✅ SCENARIO MANIFEST")
print(SCENARIO_MANIFEST[['pump_id', 'scenario', 'failure_mode', 'target_rul_now', 'target_state_now']].to_string(index=False))

print(f"\n📋 VALIDATION SUMMARY AT NOW (Day 45 = {NOW_TS.date()})")
now_check = get_state_at_timestamp(demo_final, NOW_TS)
now_check['risk_level'] = now_check.apply(compute_risk_level, axis=1)
print(f"  Risk distribution: {dict(now_check['risk_level'].value_counts())}")

# Check if we meet targets
targets_met = True
issues = []
now_risks = now_check['risk_level'].value_counts().to_dict()
if now_risks.get('GREEN', 0) != 41:
    targets_met = False
    issues.append(f"GREEN: {now_risks.get('GREEN', 0)} vs target 41")
if now_risks.get('YELLOW', 0) != 7:
    targets_met = False
    issues.append(f"YELLOW: {now_risks.get('YELLOW', 0)} vs target 7")
if now_risks.get('RED', 0) != 2:
    targets_met = False
    issues.append(f"RED: {now_risks.get('RED', 0)} vs target 2")
if now_risks.get('FAILED', 0) != 0:
    targets_met = False
    issues.append(f"FAILED: {now_risks.get('FAILED', 0)} vs target 0")

if targets_met:
    print(f"\n🎯 ALL TARGETS MET!")
else:
    print(f"\n⚠️ TARGET MISMATCHES:")
    for issue in issues:
        print(f"    - {issue}")

print(f"\n🚀 NEXT STEPS:")
print(f"  1. Review validation report above")
print(f"  2. If targets met, swap tables:")
print(f"     ALTER TABLE RAW.PUMP_TELEMETRY SWAP WITH RAW.PUMP_TELEMETRY_V2;")
print(f"     ALTER TABLE ANALYTICS.PREDICTIONS SWAP WITH ANALYTICS.PREDICTIONS_V2;")